In [2]:
from collections.abc import Callable
import os
import pandas as pd
from statcast import FetchStatcast, subset_for_analysis, mask_k_type

In [3]:
import statcast, pybaseball

In [4]:
fetch_statcast = FetchStatcast('../statcast_data')

In [6]:
pwd


'/home/lenhart/Repos/fantasy-baseball-draft/statcast'

In [9]:
# fetch all data from 2024 regular season. The early "Seoul Series" is omitted.
df = fetch_statcast.statcast("2024-03-28", "2024-06-21", preprocess=lambda x: x).drop_duplicates()

In [ ]:
needed = ['pfx_x', 'pfx_z', 'release_spin_rate', 'release_speed', 'home_team']
df[needed]
#[col for col in df.columns if 'spin' in col]
import pathlib
path = pathlib.Path("C:\\Users\\lenha\\Repos\\ua_ms_ds\\phi_msds\\info_526\\assignments\\data").as_posix()
df[needed].to_csv(os.path.join(path, 'pitch_spin.csv'))

In [ ]:
df.home_team.unique()

In [ ]:
import pathlib
path = pathlib.Path("C:\\Users\\lenha\\Repos\\ua_ms_ds\\phi_msds\\info_526\\assignments").as_posix()
import os
result = (

    pd.read_csv(os.path.join(path, 'all_teams_data.csv'))
    .dropna()
    .drop(columns=['Team'])
    .groupby('Season')
    .agg('sum')
    #.to_csv(os.path.join(path, 'data/team_data.csv'))
)
result

In [ ]:
df[(df.hit_location == 7) | (df.hit_location == 9)][['hit_location', 'hc_x', 'hc_y']]
df.groupby('events')[['hc_x', 'hc_y']].agg(['min', 'max'])
df[df.hc_x < 90].groupby(['hit_location']).agg({'hc_x': 'mean'})

In [ ]:
df[(df['home_team'] == 'BOS') & (df['game_type'] == 'R')]

In [ ]:
df.info()

In [ ]:
df['events'].unique()
df['description'].unique()

def filter_to_needed(df):
    relevant = df.query("game_type == 'R' and description == 'hit_into_play'") # filter to regular season games, balls in play.
    relevant = relevant.query("launch_angle > 20 and launch_angle < 45 and launch_speed > 90") # filter to hard hit balls in play.

    bos_home_runs = relevant.query("home_team == 'BOS' and events == 'home_run'")
    n_samples = bos_home_runs.shape[0]  # use all home runs in boston


    boston_in_play = (
        relevant.query("home_team == 'BOS' and events != 'home_run'")
        .sample(n_samples, random_state=1347))
    other_in_play = (
        relevant.query("home_team != 'BOS' and events != 'home_run'")
        .sample(n_samples, random_state=92345))
    other_home_runs = (
        relevant.query("home_team != 'BOS' and events == 'home_run'")
        .sample(n_samples, random_state=4837))
    need_columns = ['home_team', 'launch_angle', 'launch_speed', 'events', 'stand']
    return pd.concat([df[need_columns] for df in [boston_in_play, bos_home_runs, other_in_play, other_home_runs]])

    print(relevant.shape, bos_home_runs.shape)
    # df = df[df['description'] == 'hit_into_play']
    # relevant_cols = ['launch_angle', 'launch_speed']

    # return df[relevant_cols]

hr_data = filter_to_needed(df)


In [ ]:
# https://www.fangraphs.com/leaders/major-league?pos=all&stats=bat&lg=all&qual=0&type=8&month=0&ind=1&team=0%2Cts&rost=0&age=&filter=&players=0&startdate=&enddate=&season1=1947&season=2024
import seaborn as sns
hr_data = hr_data.assign(
    home_team=lambda df: df.home_team.where(lambda x: x == 'BOS', 'Other'),
    events=lambda df: df.events.where(lambda x: x == 'home_run', 'Ball In Play'))
sns.scatterplot(hr_data, x='launch_angle', y='launch_speed', hue='home_team', style='events', alpha=.5)

In [ ]:
from statcast import is_k_looking
df.description.unique()
df.loc[is_k_looking(df)]
df.events.mask(is_k_looking(df), 'CALLED K').value_counts()


## Batter Pitcher Match Ups

In [10]:
data = (
    df
    .pipe(subset_for_analysis)
    .assign(events=mask_k_type)
    .astype({'description': 'category'})
)
data.to_parquet('../statcast/batter_pitcher_matchups.parquet')

In [ ]:
data.events.value_counts()

In [ ]:
drop = ['release_speed', 'release_pos_x',
       'release_pos_z',  'n_thruorder_pitcher', 'n_priorpa_thisgame_player_at_bat',
       'pitcher_days_since_prev_game', 'batter_days_since_prev_game',
       'pitcher_days_until_next_game', 'batter_days_until_next_game',
       'api_break_z_with_gravity', 'api_break_x_arm', 'api_break_x_batter_in',
       'arm_angle', 'spin_dir', 'spin_rate_deprecated',
       'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des',
       'game_type', 'spin_dir', 'spin_rate_deprecated',
       'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des',
       'game_type', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3', 'fielder_4',
       'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9',
       'release_pos_y', 'estimated_ba_using_speedangle',
       'estimated_woba_using_speedangle', 'woba_value', 'woba_denom',
       'babip_value', 'iso_value', 'launch_speed_angle', 'at_bat_number',
       'pitch_number', 'pitch_name', 'home_score', 'away_score', 'bat_score',
       'fld_score', 'post_away_score', 'post_home_score', 'post_bat_score',
       'post_fld_score', 'if_fielding_alignment', 'of_fielding_alignment',
       'spin_axis', 'delta_home_win_exp', 'delta_run_exp', 'bat_speed',
       'swing_length', 'estimated_slg_using_speedangle',
       'delta_pitcher_run_exp', 'hyper_speed', 'home_score_diff',
       'bat_score_diff', 'home_win_exp', 'bat_win_exp', 'age_pit_legacy',
       'age_bat_legacy', 'age_pit', 'age_bat', 'stand', 'p_throws', 'home_team', 'away_team',]
df.drop(columns=drop)


In [ ]:
def add_at_bat_id(df):
    """Encode at bat as 10 bit shift of game id plus at-bat number."""
    return (df.game_pk.to_numpy() << 10) + df.at_bat_number

at_bat = add_at_bat_id(df)


In [ ]:

def merge_to_1_1_and_following(df: pd.DataFrame) -> pd.DataFrame:
    df = df[['game_date', 'game_pk', 'batter', 'pitcher', 'at_bat_number', 'pitch_number', 'pitch_type', 'balls', 'strikes']].copy()
    df['at_bat_id'] = add_at_bat_id(df)
    has_1_strike = df.strikes == 1
    has_0_strike = df.strikes == 0
    has_1_ball = df.balls == 1
    has_0_ball = df.balls == 0
    is_1_1_count = has_1_strike & has_1_ball
    can_lead_to_1_1 = (has_1_strike & has_0_ball) | (has_0_strike & has_1_ball)
    lhs = df.loc[is_1_1_count, :]
    rhs = df.loc[can_lead_to_1_1, ['pitch_type', 'at_bat_id', 'pitch_number']]
    df = lhs.merge(rhs, on='at_bat_id', how='left', suffixes=('_in_one_one', '_leading'))
    return df.query('pitch_number_leading == pitch_number_in_one_one - 1')

results = merge_to_1_1_and_following(df)#.duplicated()


In [ ]:
results.head()

In [ ]:
pitch_type_map = {
    'CH': 'Changeup',
    'CU': 'Curveball',
    'EP': "Eephus",
    'FA': 'Fastball',
    'FC': 'Cutter',
    'FF': "Four-seam Fastball",
    'FO': "Forkball",
    'FS': "Splitter",
    'KC': "Knuckle-curve",
    'KN': "Knuckle-ball",
    'PO': "Pickoff",
    'SC': "Screwball",
    'SI': "Sinker",
    'SL': "Slider",
    'ST': "Sweeper",
    'SV': "Slurve"
}

In [ ]:
results_clean = (
    results[['game_pk', 'batter', 'pitcher', 'at_bat_number', 'pitch_type_leading', 'pitch_type_in_one_one']]
    .assign(
        pitch_type_leading=lambda df: df.pitch_type_leading.map(pitch_type_map),
        pitch_type_in_one_one=lambda df: df.pitch_type_in_one_one.map(pitch_type_map))
)
results_clean.groupby('pitch_type_leading').describe()

In [ ]:
file_name = "pitch_selection_in_one_one_counts.csv"
results_clean.to_csv(file_name)

In [ ]:
directory = os.path.normpath("C:/Users/lenha/Repos/ua_ms_ds/phi_msds/info_526/assignments/data")
path = os.path.join(directory, file_name)
results_clean.to_csv(path)